# LangChain 2026 — COMPLETE Master Notebook
## Fundamentals → LCEL → Data/RAG → Agents → LangGraph → Deep Agents → Production

**Docs reviewed: 2026-10-04**

This notebook intentionally covers the **whole modern LangChain ecosystem**, not only agents.

### Coverage
- ecosystem/package architecture
- Documents
- chat models + model factory + profiles + streaming/batching
- messages + multimodal/content blocks
- prompts + placeholders + partials
- output parsers + native structured output
- Runnable interface + LCEL
- retries/fallbacks/config/caching/rate limiting
- document loaders
- text splitters
- embeddings
- vector stores
- retrievers + reranking patterns
- 2-step RAG + hybrid/agentic RAG
- tools + manual tool calling + ToolRuntime
- create_agent + structured agent output
- middleware + context engineering
- short/long-term memory
- streaming + events
- guardrails + HITL
- MCP
- multi-agent patterns
- LangGraph Graph/Functional API concepts, persistence, interrupts, subgraphs
- Deep Agents
- LangSmith testing/evals/observability
- legacy/classic migration
- production architecture + end-to-end examples

> Network/paid calls are guarded with `RUN_LIVE = False`.

## Modern ecosystem mental model

```text
langchain-core
  ├─ Document
  ├─ Messages
  ├─ Prompts
  ├─ Runnable / LCEL
  ├─ Output Parsers
  ├─ Retriever / VectorStore interfaces
  └─ callbacks/config/cache abstractions

Provider + community integrations
  ├─ Chat Models
  ├─ Embeddings
  ├─ Document Loaders
  ├─ Vector Stores
  ├─ Retrievers
  └─ Tools

langchain
  ├─ init_chat_model
  ├─ init_embeddings
  ├─ tools
  ├─ agents
  └─ middleware/runtime APIs

LangGraph → durable orchestration
Deep Agents → batteries-included long-running agents
LangSmith → tracing/testing/evaluation/Studio/deployment
```

# 1. Package architecture

| Package | Role |
|---|---|
| `langchain` | streamlined high-level APIs |
| `langchain-core` | core abstractions and LCEL |
| `langchain-community` | community integrations |
| provider packages | OpenAI/Anthropic/Google/AWS/etc. integrations |
| `langchain-text-splitters` | splitting/chunking |
| `langchain-classic` | legacy chains/retrievers/indexing |
| `langgraph` | graph runtime/orchestration |
| `deepagents` | long-running agent harness |
| `langsmith` | tracing/evals/testing/deployment tooling |

**v1 change:** main `langchain` namespace was deliberately reduced. This made it look agent-centric, but retrieval/data components remain active across `langchain-core` and integration packages.

# 2. Installation

In [ ]:
# Uncomment in a fresh environment:
# %pip install -U langchain langchain-core langchain-community langchain-text-splitters
# %pip install -U langchain-openai langgraph deepagents langchain-mcp-adapters pydantic

Typical provider installs:

```bash
pip install -U langchain-openai
pip install -U langchain-anthropic
pip install -U langchain-google-genai
pip install -U langchain-aws
pip install -U langchain-huggingface
pip install -U langchain-ollama
```

In [ ]:
import os
RUN_LIVE = False
CHAT_MODEL = os.getenv("LANGCHAIN_MODEL", "openai:gpt-5.4-mini")
EMBED_MODEL = os.getenv("LANGCHAIN_EMBED_MODEL", "openai:text-embedding-3-small")

# 3. `Document` — retrieval data unit

In [ ]:
from langchain_core.documents import Document

doc = Document(
    id="doc-001",
    page_content="LangChain provides composable building blocks.",
    metadata={"source": "course_notes", "chapter": 1},
)
print(doc.page_content)
print(doc.metadata)
print(doc.id)

`Document` is for retrieval/data processing, not chat I/O. Important fields are `page_content`, `metadata`, and optional `id`.

Useful metadata: source, page, section, timestamp, tenant, ACL/access level, document type.

# 4. Chat Models

In [ ]:
from langchain.chat_models import init_chat_model
model = init_chat_model(CHAT_MODEL)
type(model)

Chat models are Runnables. Core execution methods:
- `invoke` / `ainvoke`
- `batch` / `abatch`
- `stream` / `astream`

Provider capabilities may include tool calling, structured output, multimodal input, reasoning blocks, token usage and logprobs.

In [ ]:
if RUN_LIVE:
    response = model.invoke("Explain semantic search in two sentences.")
    print(type(response), response.content)

In [ ]:
if RUN_LIVE:
    responses = model.batch([
        "Define embeddings.",
        "Define vector store.",
        "Define retriever.",
    ])
    for r in responses:
        print(r.content)

In [ ]:
if RUN_LIVE:
    for chunk in model.stream("Explain LCEL briefly."):
        print(chunk.content, end="")

## 4.1 Model profiles

In [ ]:
getattr(model, "profile", None)

Profiles can expose model capability metadata such as context-window information, tool calling and structured-output support. Provider-specific classes remain useful when advanced provider settings are required.

# 5. Messages

In [ ]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
messages = [
    SystemMessage(content="You are a concise teacher."),
    HumanMessage(content="What is a vector store?"),
]
messages

## 5.1 Standard content blocks / multimodal

In [ ]:
msg = HumanMessage(content_blocks=[
    {"type": "text", "text": "Describe this image."},
    {"type": "image", "url": "https://example.com/image.jpg"},
])
msg.content_blocks

Content blocks normalize supported provider content such as text, images, files, audio, reasoning and tool-related content. Use messages for conversation; use `Document` for retrieval corpora.

## 5.2 Message history trimming

In [ ]:
from langchain_core.messages import trim_messages
history = [HumanMessage(content="Hello"), AIMessage(content="Hi"), HumanMessage(content="Teach RAG")]
# Use trim_messages(...) with your token/length policy for large histories.

# 6. Prompt Templates

In [ ]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
prompt = PromptTemplate.from_template("Explain {topic} to a {level} learner.")
print(prompt.format(topic="embeddings", level="beginner"))

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI teacher. Explain at {level} level."),
    ("human", "{question}"),
])
chat_prompt.invoke({"level":"advanced", "question":"What is LCEL?"})

## 6.1 `MessagesPlaceholder`

In [ ]:
history_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a tutor."),
    MessagesPlaceholder("history"),
    ("human", "{question}"),
])
history_prompt.invoke({
    "history": [HumanMessage(content="I am learning LangChain."), AIMessage(content="Great.")],
    "question": "Teach retrievers.",
})

## 6.2 Partial prompts

In [ ]:
partial = chat_prompt.partial(level="expert")
partial.invoke({"question":"Explain MMR."})

Prompt formats include f-string, mustache and Jinja2. Do not accept arbitrary untrusted Jinja templates as safe executable content.

# 7. Output Parsers vs Native Structured Output

Modern rule:

```text
Native structured output supported? → prefer it
Otherwise / extra parsing needed? → output parser remains useful
```

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser
from pydantic import BaseModel, Field

string_parser = StrOutputParser()
json_parser = JsonOutputParser()

class TopicSummary(BaseModel):
    topic: str
    explanation: str
    difficulty: str

pydantic_parser = PydanticOutputParser(pydantic_object=TopicSummary)
print(pydantic_parser.get_format_instructions()[:400])

Other parser families include lists, Markdown/numbered lists, XML, JSON/Pydantic and tool/function-output parsers.

## 7.1 Model-native structured output

In [ ]:
class CityDetails(BaseModel):
    city: str
    country: str
    population_note: str | None = None

structured_model = model.with_structured_output(CityDetails)
if RUN_LIVE:
    print(structured_model.invoke("Give details for Bengaluru."))

# 8. Runnable Interface and LCEL

A `Runnable` is a composable unit with sync, async, batch and streaming interfaces. LCEL composes Runnables declaratively, commonly with `|`.

In [ ]:
from langchain_core.runnables import RunnableLambda
plus_one = RunnableLambda(lambda x: x + 1)
double = RunnableLambda(lambda x: x * 2)
sequence = plus_one | double
sequence.invoke(10)

## 8.1 Prompt → Model → Parser

In [ ]:
llm_chain = chat_prompt | model | StrOutputParser()
if RUN_LIVE:
    print(llm_chain.invoke({"level":"beginner", "question":"What is a Runnable?"}))

## 8.2 `RunnableParallel`

In [ ]:
from langchain_core.runnables import RunnableParallel
parallel = RunnableParallel(
    doubled=RunnableLambda(lambda x: x*2),
    squared=RunnableLambda(lambda x: x**2),
)
parallel.invoke(5)

## 8.3 Dict syntax = parallel branches

In [ ]:
pipe = RunnableLambda(lambda x: x+1) | {
    "times_2": RunnableLambda(lambda x: x*2),
    "times_5": RunnableLambda(lambda x: x*5),
}
pipe.invoke(1)

## 8.4 `RunnablePassthrough`

In [ ]:
from langchain_core.runnables import RunnablePassthrough
passthrough = RunnablePassthrough.assign(length=lambda x: len(x["text"]))
passthrough.invoke({"text":"LangChain"})

## 8.5 `RunnableBranch`

In [ ]:
from langchain_core.runnables import RunnableBranch
branch = RunnableBranch(
    (lambda x: x < 0, RunnableLambda(lambda x: "negative")),
    (lambda x: x == 0, RunnableLambda(lambda x: "zero")),
    RunnableLambda(lambda x: "positive"),
)
[branch.invoke(x) for x in [-2,0,3]]

Important Runnable methods:
- invoke / ainvoke
- batch / abatch / *_as_completed
- stream / astream / astream_events
- with_retry / with_fallbacks / with_config / with_types / with_listeners
- configurable_fields / configurable_alternatives
- get_input_schema / get_output_schema / get_graph

In [ ]:
unreliable = RunnableLambda(lambda x: x)
retriable = unreliable.with_retry(stop_after_attempt=3)
primary = RunnableLambda(lambda x: f"primary:{x}")
fallback = RunnableLambda(lambda x: f"fallback:{x}")
resilient = primary.with_fallbacks([fallback])

## 8.6 Configurable alternatives

In [ ]:
from langchain_core.runnables.utils import ConfigurableField
# provider_model.configurable_alternatives(ConfigurableField(id="llm"), ...)

# 9. Cache, Rate Limit, Config and Debug

In [ ]:
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
set_llm_cache(InMemoryCache())

Model caching can reduce repeated calls. Streaming cache behavior is not identical to normal invocation; verify before relying on it.

In [ ]:
from langchain_core.rate_limiters import InMemoryRateLimiter
rate_limiter = InMemoryRateLimiter(
    requests_per_second=2,
    check_every_n_seconds=0.1,
    max_bucket_size=5,
)

`InMemoryRateLimiter` is process-local and request-rate-based, not a distributed/token-volume quota system.

In [ ]:
from langchain_core.globals import set_debug, set_verbose
# set_debug(True)
# set_verbose(True)

In [ ]:
# Runnable config supports tags, metadata and max_concurrency patterns.
# result = chain.invoke(input, config={"tags":["bootcamp"], "metadata":{"module":"lcel"}})

# 10. Document Loaders

Document loaders convert external sources to `Document`. Shared methods commonly include `load()` and `lazy_load()`.

In [ ]:
# Text
# from langchain_community.document_loaders import TextLoader
# docs = TextLoader("notes.txt", encoding="utf-8").load()

In [ ]:
# PDF
# from langchain_community.document_loaders import PyPDFLoader
# pages = PyPDFLoader("document.pdf").load()

PDF ecosystem options include PyPDF, PyMuPDF, PDFMiner, pdfplumber, Unstructured, Docling, Upstage and document-intelligence integrations. Choice affects OCR/layout/table quality.

In [ ]:
# CSV
# from langchain_community.document_loaders import CSVLoader
# docs = CSVLoader("data.csv").load()

In [ ]:
# JSON
# from langchain_community.document_loaders import JSONLoader
# docs = JSONLoader(file_path="data.json", jq_schema=".[]", text_content=False).load()

In [ ]:
# Web
# from langchain_community.document_loaders import WebBaseLoader
# docs = WebBaseLoader("https://example.com").load()

In [ ]:
# Directory
# from langchain_community.document_loaders import DirectoryLoader
# docs = DirectoryLoader("./data", glob="**/*.txt").load()

In [ ]:
# Docling
# from langchain_docling.loader import DoclingLoader
# docs = DoclingLoader(file_path="document.pdf").load()

In [ ]:
# Unstructured (package/setup depends on local/API use)
# from langchain_unstructured import UnstructuredLoader
# docs = UnstructuredLoader(file_path="document.pdf").load()

Source families include local files, S3/Blob/GCS, Drive, OneDrive/SharePoint, Slack, Notion, databases, web, APIs, dataframes, chat/email exports and notebooks.

For very large corpora, prefer lazy/streaming ingestion when supported.

# 11. Text Splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text = """LangChain provides abstractions for models, prompts and retrieval.

Retrieval systems load, split, embed and search documents."""
splitter = RecursiveCharacterTextSplitter(chunk_size=80, chunk_overlap=15)
chunks = splitter.split_text(text)
chunks

In [ ]:
docs = [Document(page_content=text, metadata={"source":"lesson"})]
chunk_docs = splitter.split_documents(docs)
[(d.page_content, d.metadata) for d in chunk_docs]

`RecursiveCharacterTextSplitter` is a strong general default. Other splitter families cover token, Markdown, HTML, JSON, code/languages, spaCy, NLTK and semantic strategies.

Chunking trade-offs:
- too small → fragmented context/noise
- too large → poor precision/high token use
- too much overlap → duplicate retrieval

# 12. Embeddings

In [ ]:
from langchain.embeddings import init_embeddings
embeddings = init_embeddings(EMBED_MODEL)

In [ ]:
if RUN_LIVE:
    qv = embeddings.embed_query("What is semantic search?")
    dvs = embeddings.embed_documents(["LangChain", "Vector search"])
    print(len(qv), len(dvs))

Typical provider families include OpenAI, Azure, Bedrock, Cohere, Hugging Face, Mistral, Ollama and Google Vertex AI.

The generic current embeddings docs are centered on **text embeddings**; do not assume universal multimodal embedding support.

# 13. Vector Stores

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
vector_store = InMemoryVectorStore(embedding=embeddings)

sample_docs = [
    Document(id="1", page_content="LCEL composes Runnables.", metadata={"topic":"lcel"}),
    Document(id="2", page_content="A retriever returns relevant Documents.", metadata={"topic":"rag"}),
    Document(id="3", page_content="Middleware intercepts agent lifecycle.", metadata={"topic":"agents"}),
]
if RUN_LIVE:
    vector_store.add_documents(sample_docs)

In [ ]:
if RUN_LIVE:
    print(vector_store.similarity_search("retrieval", k=2))
    print(vector_store.similarity_search_with_score("agent hooks", k=2))

In [ ]:
if RUN_LIVE:
    results = vector_store.similarity_search(
        "retrieval", k=3,
        filter=lambda d: d.metadata.get("topic") == "rag",
    )

In [ ]:
if RUN_LIVE:
    results = vector_store.max_marginal_relevance_search(
        "LangChain concepts", k=2, fetch_k=3, lambda_mult=0.5
    )

Common production integrations include Chroma, Qdrant, Pinecone, FAISS, Milvus, Weaviate, Elasticsearch, PostgreSQL/PGVector families, MongoDB and cloud-native vector systems.

# 14. Retrievers

A retriever is `string query → list[Document]`, and follows the standard Runnable interface.

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k":3})
if RUN_LIVE:
    docs = retriever.invoke("What is middleware?")

In [ ]:
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":2, "fetch_k":5, "lambda_mult":0.5},
)

## 14.1 Custom retriever

In [ ]:
from langchain_core.retrievers import BaseRetriever
class SimpleRetriever(BaseRetriever):
    docs: list[Document]
    k: int = 2
    def _get_relevant_documents(self, query: str) -> list[Document]:
        return self.docs[:self.k]

custom_retriever = SimpleRetriever(docs=sample_docs)
custom_retriever.invoke("anything")

Retrieval patterns include vector similarity, MMR, lexical/BM25, hybrid, multi-query, parent/child, metadata/self-query, compression, reranking and external search/database retrieval.

A common reranking pipeline is `broad top-N → reranker/compressor → small high-quality context`.

# 15. Two-Step RAG

```text
Question → Retriever → Relevant Documents → Prompt → Model → Answer
```

Use when retrieval should always happen. Benefits: predictable, easy to debug, typically stable latency.

In [ ]:
def format_docs(docs: list[Document]) -> str:
    return "\n\n".join(
        f"[source={d.metadata.get('source', d.metadata.get('topic','unknown'))}]\n{d.page_content}"
        for d in docs
    )

In [ ]:
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer only from supplied context. If absent, say you do not know."),
    ("human", "Question:\n{question}\n\nContext:\n{context}"),
])

In [ ]:
from langchain_core.runnables import RunnablePassthrough

two_step_rag = (
    {"context": retriever | RunnableLambda(format_docs), "question": RunnablePassthrough()}
    | rag_prompt
    | model
    | StrOutputParser()
)
if RUN_LIVE:
    print(two_step_rag.invoke("What is a retriever?"))

This LCEL pattern replaces many old `RetrievalQA`-style use cases without requiring legacy chain classes.

# 16. Agentic and Hybrid RAG

- **2-Step RAG:** retrieve first every time.
- **Agentic RAG:** model decides whether/when/how to retrieve.
- **Hybrid RAG:** explicit routing/query transformation/retrieval validation/reranking/post-checks.

Agentic RAG usually exposes retrieval as a tool.

# 17. Tools and Manual Tool Calling

In [ ]:
from langchain.tools import tool
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a*b
multiply.invoke({"a":6, "b":7})

In [ ]:
tool_bound_model = model.bind_tools([multiply])
if RUN_LIVE:
    ai_msg = tool_bound_model.invoke("What is 12*8?")
    print(ai_msg.tool_calls)

Manual protocol: model emits tool call → app executes → app creates ToolMessage → model sees result → final response. `create_agent()` automates this loop.

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search internal knowledge base."""
    docs = custom_retriever.invoke(query)
    return format_docs(docs)

# 18. `ToolRuntime`

In [ ]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime
@dataclass
class UserContext:
    user_id: str
    role: str = "student"

@tool
def current_user(runtime: ToolRuntime[UserContext]) -> str:
    """Return current runtime user info."""
    return f"{runtime.context.user_id}:{runtime.context.role}"

`ToolRuntime` can expose state, context, store, custom stream writer, execution info and server info without exposing those fields in the model-visible tool schema.

# 19. `create_agent()`

In [ ]:
from langchain.agents import create_agent
agent = create_agent(
    model=CHAT_MODEL,
    tools=[multiply, search_knowledge_base],
    system_prompt="You are helpful. Use tools when needed.",
)
if RUN_LIVE:
    result = agent.invoke({"messages":[{"role":"user","content":"What is 9*9?"}]})
    print(result["messages"][-1].content)

Modern agent harness = model + system prompt + tools + middleware + state/context/store + structured output + checkpointer. Current LangChain agents run on LangGraph runtime.

## 19.1 Agent structured output

In [ ]:
class ResearchAnswer(BaseModel):
    answer: str
    confidence: str
    sources_used: list[str]

structured_agent = create_agent(model=CHAT_MODEL, tools=[search_knowledge_base], response_format=ResearchAnswer)

In [ ]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy
provider_strategy_agent = create_agent(model=CHAT_MODEL, tools=[], response_format=ProviderStrategy(ResearchAnswer))
tool_strategy_agent = create_agent(model=CHAT_MODEL, tools=[], response_format=ToolStrategy(ResearchAnswer))

# 20. Middleware

Middleware is the major v1 architecture for lifecycle interception, dynamic context, model/tool control, retries, limits, guardrails and HITL.

Common hook concepts:
`before_agent`, `before_model`, `after_model`, `after_agent`, `wrap_model_call`, `wrap_tool_call`.

In [ ]:
from langchain.agents.middleware import before_model, after_model
from langgraph.runtime import Runtime
from langchain.agents import AgentState

@before_model
def log_before(state: AgentState, runtime: Runtime) -> dict | None:
    print("before model", len(state["messages"]))
    return None

@after_model
def log_after(state: AgentState, runtime: Runtime) -> dict | None:
    print("after model")
    return None

## 20.1 Dynamic prompt

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest
@dynamic_prompt
def role_prompt(request: ModelRequest) -> str:
    role = request.runtime.context.role
    return f"You are an instructor. Adapt for role: {role}."

## 20.2 Class middleware and dynamic model routing

In [ ]:
from langchain.agents.middleware import AgentMiddleware
from typing import Callable
from langchain.agents.middleware.types import ModelResponse

cheap_model = init_chat_model("openai:gpt-5.4-mini")
strong_model = init_chat_model(CHAT_MODEL)

class ModelRouter(AgentMiddleware):
    def wrap_model_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
        chosen = strong_model if len(request.state["messages"]) > 10 else cheap_model
        return handler(request.override(model=chosen))

## 20.3 Prebuilt middleware

In [ ]:
from langchain.agents.middleware import (
    SummarizationMiddleware, HumanInTheLoopMiddleware,
    ModelCallLimitMiddleware, ToolCallLimitMiddleware,
    ModelFallbackMiddleware, PIIMiddleware, TodoListMiddleware,
    LLMToolSelectorMiddleware, ToolRetryMiddleware, ModelRetryMiddleware,
    ContextEditingMiddleware, ClearToolUsesEdit,
)

In [ ]:
summarized_agent = create_agent(
    model=CHAT_MODEL, tools=[],
    middleware=[SummarizationMiddleware(model="openai:gpt-5.4-mini", trigger=("tokens",4000), keep=("messages",20))],
)

In [ ]:
pii_agent = create_agent(
    model=CHAT_MODEL, tools=[],
    middleware=[PIIMiddleware("email", strategy="redact", apply_to_input=True)],
)

In [ ]:
limited_agent = create_agent(
    model=CHAT_MODEL, tools=[multiply],
    middleware=[
        ModelCallLimitMiddleware(run_limit=5, exit_behavior="end"),
        ToolCallLimitMiddleware(tool_name="multiply", run_limit=3),
    ],
)

In [ ]:
retry_agent = create_agent(
    model=CHAT_MODEL, tools=[multiply],
    middleware=[ToolRetryMiddleware(max_retries=2), ModelRetryMiddleware(max_retries=2)],
)

In [ ]:
selector_agent = create_agent(
    model=CHAT_MODEL, tools=[multiply, search_knowledge_base],
    middleware=[LLMToolSelectorMiddleware(model="openai:gpt-5.4-mini", max_tools=2)],
)

In [ ]:
context_edit_agent = create_agent(
    model=CHAT_MODEL, tools=[multiply],
    middleware=[ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=2000, keep=3)])],
)

Also know: tool-error handling, tool emulation, provider tool search, shell/file-search/virtual-filesystem patterns. Some are provider/version/environment dependent.

# 21. Context Engineering, State and Memory

| Concept | Scope |
|---|---|
| Context | static run-specific dependencies |
| State | mutable thread/workflow state |
| Store | durable cross-thread memory |

In [ ]:
@dataclass
class TeachingContext:
    user_id: str
    level: str
context_agent = create_agent(model=CHAT_MODEL, tools=[], context_schema=TeachingContext)

## 21.1 Short-term memory / checkpointer

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
memory_agent = create_agent(model=CHAT_MODEL, tools=[], checkpointer=InMemorySaver())
thread_config = {"configurable":{"thread_id":"student-001"}}

## 21.2 Long-term Store

In [ ]:
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()
store.put(("users","student-001"), "prefs", {"style":"technical but simple"})
store.get(("users","student-001"), "prefs").value

Production should use persistent checkpointer/store backends, not only in-memory implementations.

# 22. Streaming and Event Streaming

In [ ]:
if RUN_LIVE:
    for chunk in agent.stream({"messages":[{"role":"user","content":"Explain tools."}]}, stream_mode="messages"):
        print(chunk)

In [ ]:
# async for event in llm_chain.astream_events({"level":"beginner", "question":"Explain LCEL"}):
#     print(event["event"], event.get("name"))

Streaming may expose model chunks, state updates, tool progress, custom events and nested subgraph/subagent activity.

# 23. Guardrails and Human-in-the-Loop

Guardrails can be deterministic (regex/rules/schema/permissions) or model-based (classifier/policy/quality judge). Apply them at input, model, tool and output layers.

In [ ]:
@tool
def delete_record(record_id: str) -> str:
    """Delete a record. Demo only."""
    return f"Deleted {record_id}"

approval_agent = create_agent(
    model=CHAT_MODEL,
    tools=[delete_record],
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(interrupt_on={
        "delete_record":{"allowed_decisions":["approve","edit","reject"]}
    })],
)

# 24. MCP — Model Context Protocol

MCP provides a standardized bridge to external tool servers. Key concepts: multiple servers, stdio, HTTP, authentication, sessions, interceptors and server-provided capabilities.

In [ ]:
# from langchain_mcp_adapters.client import MultiServerMCPClient
# client = MultiServerMCPClient({
#   "math":{"transport":"stdio","command":"python","args":["/path/math_server.py"]},
#   "remote":{"transport":"http","url":"http://localhost:8000/mcp"},
# })
# tools = await client.get_tools()
# mcp_agent = create_agent(CHAT_MODEL, tools)

In [ ]:
# Server concept:
# from fastmcp import FastMCP
# mcp = FastMCP("Demo")
# @mcp.tool()
# def add_numbers(a:int,b:int)->int: return a+b
# mcp.run(transport="stdio")

# 25. Multi-Agent Patterns

| Pattern | Best fit |
|---|---|
| Subagents | supervisor + isolated specialists |
| Handoffs | active conversational control moves |
| Skills | one agent loads expertise on demand |
| Router | classify/fan-out to specialists |
| Custom workflow | exact LangGraph topology |

Do not use multi-agent by default; single-agent + good context/tool selection can be simpler.

# 26. LangGraph

Use LangGraph for explicit topology, deterministic+agentic workflows, branching, parallelism, persistence, interrupts, subgraphs, time travel and fault tolerance.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class GraphState(TypedDict):
    text: str
    normalized: str

def normalize(state: GraphState):
    return {"normalized": state["text"].strip().lower()}

b = StateGraph(GraphState)
b.add_node("normalize", normalize)
b.add_edge(START, "normalize")
b.add_edge("normalize", END)
graph = b.compile()
graph.invoke({"text":"  LANGCHAIN  ", "normalized":""})

## 26.1 Conditional edges

In [ ]:
class NumberState(TypedDict):
    number: int
    result: str

def route(state: NumberState): return "even" if state["number"]%2==0 else "odd"
def even_node(state: NumberState): return {"result":"even"}
def odd_node(state: NumberState): return {"result":"odd"}

gb=StateGraph(NumberState)
gb.add_node("route_node", lambda state: {})
gb.add_node("even", even_node); gb.add_node("odd", odd_node)
gb.add_edge(START,"route_node")
gb.add_conditional_edges("route_node", route, {"even":"even","odd":"odd"})
gb.add_edge("even",END); gb.add_edge("odd",END)
number_graph=gb.compile()
number_graph.invoke({"number":5,"result":""})

## 26.2 Persistence + Store

In [ ]:
persistent_graph = b.compile(checkpointer=InMemorySaver(), store=InMemoryStore())

## 26.3 Interrupt / resume

In [ ]:
from langgraph.types import interrupt, Command
class ReviewState(TypedDict):
    draft: str
    approved: bool

def review(state: ReviewState):
    decision = interrupt({"question":"Approve?", "draft":state["draft"]})
    return {"approved": bool(decision)}

rb=StateGraph(ReviewState); rb.add_node("review",review); rb.add_edge(START,"review"); rb.add_edge("review",END)
review_graph=rb.compile(checkpointer=InMemorySaver())
# config={"configurable":{"thread_id":"review-1"}}
# review_graph.invoke({"draft":"hello","approved":False}, config=config)
# review_graph.invoke(Command(resume=True), config=config)

LangGraph also provides subgraphs, time travel, fault tolerance, event streaming, and a Functional API in addition to the explicit Graph API. Teach Graph API first for visibility; use Functional API when normal Python control flow is preferable.

# 27. Deep Agents

In [ ]:
from deepagents import create_deep_agent
deep_agent = create_deep_agent(model=CHAT_MODEL, tools=[multiply], system_prompt="You are a careful knowledge-work agent.")

Deep Agents targets long-running/context-heavy work such as research, coding and analysis.

Key areas:
- tools
- virtual filesystem / context offloading
- backends and permissions
- sandboxes/interpreters
- multimodal
- streaming/event streaming
- skills
- memory
- retrieval
- context engineering
- profiles
- subagents, dynamic/async subagents
- HITL
- grading rubrics
- MCP
- ACP / A2A
- fault tolerance / production

Filesystem is also a context-engineering strategy: offload huge artifacts to files, retain path/summary in model context, and read only relevant slices later.

# 28. LangSmith — Observability, Testing and Evals

In [ ]:
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "..."
# os.environ["LANGSMITH_PROJECT"] = "langchain-master-course"

Trace: prompts, model calls, tools, retrieval, nested graphs, latency, tokens and errors.

Evaluate: correctness, groundedness, retrieval quality, tool selection/arguments, trajectory, task success, safety, latency and cost.

Testing docs include unit tests, integration tests and agent evaluations.

# 29. Legacy / `langchain-classic` Migration

Legacy examples may use `LLMChain`, `ConversationChain`, old memory classes, old retrievers and `AgentExecutor` patterns. Many moved to `langchain-classic`.

| Old | Modern direction |
|---|---|
| `LLMChain` | `prompt | model | parser` |
| many old Chains | LCEL / LangGraph / classic compatibility |
| `ConversationChain` | agent + state/checkpointer |
| old memory classes | state/checkpointer/store/middleware |
| `create_react_agent` | `create_agent` |
| agent `prompt=` | `system_prompt=` |
| pre/post model hooks | middleware |
| injected state helpers | `ToolRuntime` |
| prompted JSON | native/agent structured output |
| giant tool list | tool selection/search/subagents |

# 30. Choosing the Right Abstraction

- **Plain model:** one generation call.
- **Prompt + model + parser:** deterministic linear generation.
- **LCEL:** composable deterministic pipeline.
- **Retriever:** query → Documents.
- **2-step RAG:** retrieval is always required.
- **Agent:** model chooses actions/tools.
- **LangGraph:** exact workflow/state topology matters.
- **Deep Agents:** long-running context-heavy knowledge worker.

# 31. End-to-End A — Semantic Search

In [ ]:
demo_docs=[
    Document(id="a",page_content="LCEL composes Runnable components.",metadata={"chapter":"lcel"}),
    Document(id="b",page_content="A vector store supports semantic similarity search.",metadata={"chapter":"retrieval"}),
    Document(id="c",page_content="Agent middleware intercepts lifecycle operations.",metadata={"chapter":"agents"}),
]
demo_splitter=RecursiveCharacterTextSplitter(chunk_size=80,chunk_overlap=10)
demo_chunks=demo_splitter.split_documents(demo_docs)
demo_store=InMemoryVectorStore(embedding=embeddings)
if RUN_LIVE:
    demo_store.add_documents(demo_chunks)
    print(demo_store.similarity_search("semantic retrieval",k=2))

# 32. End-to-End B — Two-Step RAG

In [ ]:
demo_retriever=demo_store.as_retriever(search_kwargs={"k":3})
demo_rag=(
    {"context":demo_retriever|RunnableLambda(format_docs),"question":RunnablePassthrough()}
    | ChatPromptTemplate.from_messages([
        ("system","Answer only from context. If insufficient, say so."),
        ("human","Question: {question}\n\nContext:\n{context}"),
    ])
    | model | StrOutputParser()
)
if RUN_LIVE:
    print(demo_rag.invoke("What does middleware do?"))

# 33. End-to-End C — Agentic RAG

In [ ]:
@tool
def retrieve_course_context(query: str) -> str:
    """Retrieve relevant course context."""
    return format_docs(demo_retriever.invoke(query))

rag_agent=create_agent(
    model=CHAT_MODEL,
    tools=[retrieve_course_context,multiply],
    system_prompt="Use retrieval when course knowledge is required.",
)

# 34. End-to-End D — Production-Oriented Agent

In [ ]:
@dataclass
class ProductionContext:
    user_id: str
    level: str

class FinalResponse(BaseModel):
    answer: str
    used_retrieval: bool
    safety_note: str | None = None

@dynamic_prompt
def production_prompt(request: ModelRequest) -> str:
    return f"You are a reliable enterprise assistant. User level={request.runtime.context.level}. Use retrieval when needed; do not invent evidence."

production_agent=create_agent(
    model=CHAT_MODEL,
    tools=[retrieve_course_context,multiply],
    context_schema=ProductionContext,
    response_format=FinalResponse,
    checkpointer=InMemorySaver(),
    store=InMemoryStore(),
    middleware=[
        production_prompt,
        PIIMiddleware("email",strategy="redact",apply_to_input=True),
        ToolRetryMiddleware(max_retries=2),
        ModelRetryMiddleware(max_retries=2),
        ModelCallLimitMiddleware(run_limit=8,exit_behavior="end"),
    ],
)

# 35. Production Checklist

### Data / RAG
- parser/loader quality
- metadata/ACL strategy
- chunking evaluation
- embedding fit
- vector filtering/hybrid search
- retrieval k + reranking
- citations/provenance

### Model / Runnable
- capability + context window
- timeout/retry/rate limit/cache
- structured output
- fallbacks

### Agent
- precise tools and schemas
- permissions/HITL
- model/tool call limits
- context growth + summarization

### Memory / Security
- thread vs cross-thread scope
- retention + tenant isolation
- secrets outside prompts
- PII controls
- sandbox shell/code

### Reliability / Evaluation
- idempotent side effects
- durable persistence
- graceful partial failures
- traces + datasets + regression evals

# 36. Complete Teaching Roadmap

1. Ecosystem/packages
2. Document
3. Models
4. Messages/content blocks
5. Prompts
6. Output parsers
7. Native structured output
8. Runnable interface
9. LCEL sequence/parallel/branch/passthrough
10. Config/retry/fallback/cache/rate-limit
11. Document loaders
12. Text splitters
13. Embeddings
14. Vector stores
15. Retrievers
16. Reranking/hybrid retrieval
17. Two-step RAG
18. Hybrid RAG
19. Tools
20. Manual tool calling
21. ToolRuntime
22. create_agent
23. Agent structured output
24. Middleware
25. Context engineering
26. Short-term memory
27. Long-term memory
28. Streaming/events
29. Guardrails
30. HITL
31. MCP
32. Multi-agent
33. LangGraph Graph API
34. Persistence/interrupts/subgraphs
35. Functional API
36. Deep Agents
37. LangSmith/tracing
38. Testing/evals
39. Legacy migration
40. Production architecture

# 37. Whole Pipeline Cheat Sheet

```text
Source
 ↓
Document Loader
 ↓
Document[]
 ↓
Text Splitter
 ↓
Chunks
 ↓
Embeddings
 ↓
Vector Store
 ↓
Retriever
 ↓
 ├─ Two-Step RAG → Prompt → Model → Parser
 └─ Retriever Tool → create_agent → Agentic RAG

Complex exact workflow → LangGraph
Long-running knowledge worker → Deep Agents
Tracing/evals/testing → LangSmith
```

# 38. Current Official Documentation Surface

Current docs index includes dedicated sections for LangChain agents/models/messages/tools/memory/streaming/structured output/middleware/guardrails/runtime/context engineering/MCP/multi-agent/retrieval/testing/deploy/observability, plus integration catalogs for chat models, tools, middleware, sandboxes, backends, checkpointers, long-term-memory stores, retrievers, splitters, embeddings, vector stores and document loaders. It separately documents LangGraph and Deep Agents.

# 39. Official Sources Reviewed

- https://docs.langchain.com/llms.txt
- https://docs.langchain.com/oss/python/langchain/overview
- https://docs.langchain.com/oss/python/migrate/langchain-v1
- https://docs.langchain.com/oss/python/langchain/models
- https://docs.langchain.com/oss/python/langchain/messages
- https://docs.langchain.com/oss/python/langchain/tools
- https://docs.langchain.com/oss/python/langchain/retrieval
- https://docs.langchain.com/oss/python/langchain/structured-output
- https://docs.langchain.com/oss/python/langchain/middleware/overview
- https://docs.langchain.com/oss/python/langchain/middleware/built-in
- https://docs.langchain.com/oss/python/langchain/runtime
- https://docs.langchain.com/oss/python/langchain/context-engineering
- https://docs.langchain.com/oss/python/langchain/short-term-memory
- https://docs.langchain.com/oss/python/langchain/long-term-memory
- https://docs.langchain.com/oss/python/langchain/mcp
- https://docs.langchain.com/oss/python/langchain/multi-agent
- https://docs.langchain.com/oss/python/langgraph/overview
- https://docs.langchain.com/oss/python/langgraph/persistence
- https://docs.langchain.com/oss/python/langgraph/interrupts
- https://docs.langchain.com/oss/python/deepagents/overview
- https://reference.langchain.com/python/langchain-core
- https://reference.langchain.com/python/langchain-text-splitters
- https://reference.langchain.com/python/langchain/embeddings
- https://docs.langchain.com/oss/python/integrations/providers/overview
- https://docs.langchain.com/oss/python/integrations/document_loaders
- https://docs.langchain.com/oss/python/integrations/embeddings
- https://docs.langchain.com/oss/python/integrations/vectorstores
- https://docs.langchain.com/oss/python/integrations/retrievers

**Note:** LangChain changes quickly. Re-check beta/recent APIs before production releases.